RAG Pipelines- Data Ingestion to vector DB pipeline

In [63]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path


In [64]:
### READ ALL pdf's inside the directory

def process_all_pdfs(pdf_directory):
    ##"""Process all PDF files in a directory"""

    all_documents = []
    pdf_dir = Path(pdf_directory)

    ## find all pdf files recursively
    pdf_files = list(pdf_dir.glob("*/**.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f" \nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            ##add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f" Loaded {len(documents)} pages")
        except Exception as e:
            print(f" Error: {e}")
        
    print(f"\nTotal documents loaded: {len(all_documents)}")  
    return all_documents

##Process all pdfs in the data directory
all_pdf_documents = process_all_pdfs("../data")


Found 3 PDF files to process
 
Processing: attention_is_all_you_need_notes.pdf
 Loaded 4 pages
 
Processing: embeddings_and_vector_databases_notes.pdf
 Loaded 4 pages
 
Processing: React_Developer_Interview_Guide_Candidate_87.pdf
 Loaded 3 pages

Total documents loaded: 11


In [65]:
all_pdf_documents

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-10-28T05:56:33+00:00', 'author': 'OpenAI', 'keywords': '', 'moddate': '2026-10-28T05:56:33+00:00', 'subject': '(unspecified)', 'title': 'Attention Is All You Need', 'trapped': '/False', 'source': '..\\data\\pdf\\attention_is_all_you_need_notes.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'source_file': 'attention_is_all_you_need_notes.pdf', 'file_type': 'pdf'}, page_content='AI / ML Study Notes\nPage 1\n Attention Is All You Need\n A structured guide to the original Transformer paper and its core ideas\nHow to use these notes\nRead once for the big picture, then revisit the definitions, examples, and comparison tables. The aim is to build\nintuition first and connect the ideas to practical AI systems.\n1. The problem before Transformers\nEarlier sequence models often used recurrent neural networks (RNNs), including LSTMs and GRUs, to process\ntokens one at 

In [66]:
### text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """split documents into smaller chunk for better rag performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators=["\n\n" , "\n" , " " , ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} into {len(split_docs)} chunks")

    ##show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Matadata: {split_docs[0].metadata}")

    return split_docs



In [67]:

chunks = split_documents(all_pdf_documents)
chunks

Split 11 into 31 chunks

Example chunk:
Content: AI / ML Study Notes
Page 1
 Attention Is All You Need
 A structured guide to the original Transformer paper and its core ideas
How to use these notes
Read once for the big picture, then revisit the de...
Matadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-10-28T05:56:33+00:00', 'author': 'OpenAI', 'keywords': '', 'moddate': '2026-10-28T05:56:33+00:00', 'subject': '(unspecified)', 'title': 'Attention Is All You Need', 'trapped': '/False', 'source': '..\\data\\pdf\\attention_is_all_you_need_notes.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'source_file': 'attention_is_all_you_need_notes.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-10-28T05:56:33+00:00', 'author': 'OpenAI', 'keywords': '', 'moddate': '2026-10-28T05:56:33+00:00', 'subject': '(unspecified)', 'title': 'Attention Is All You Need', 'trapped': '/False', 'source': '..\\data\\pdf\\attention_is_all_you_need_notes.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'source_file': 'attention_is_all_you_need_notes.pdf', 'file_type': 'pdf'}, page_content='AI / ML Study Notes\nPage 1\n Attention Is All You Need\n A structured guide to the original Transformer paper and its core ideas\nHow to use these notes\nRead once for the big picture, then revisit the definitions, examples, and comparison tables. The aim is to build\nintuition first and connect the ideas to practical AI systems.\n1. The problem before Transformers\nEarlier sequence models often used recurrent neural networks (RNNs), including LSTMs and GRUs, to process\ntokens one at 

Embedding and vector store DB

In [68]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple


In [69]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformers"""

    def __init__(self , model_name:str = "all-miniLM-L6-v2"):
        """
        initialize the embedding manager

        Args:
        model_name: Huggingface model name for sentence embeddings

        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model: {self.model_name: {e}}")
            raise 


    def generateEmbeddings(self, texts: List[str]) -> np.ndarray:
            """
            Generate embeddings for a list of texts

            Args:
            texts:List of text strings to embed

            Returns:
            numpy array of embeddings with shape(len(texts),embedding_dim)
            """

            if not self.model:
                raise ValueError("Model not loaded")

            print(f"Generating embeddings for {len(texts)} texts...")
            embeddings = self.model.encode(texts, show_progress_bar=True)
            print(f"Generated embeddings with shape: {embeddings.shape}")
            return embeddings


    def get_embedding_dimension(self) -> int:
            """Get the embedding dimension of the model"""
            if not self.model:
                raise ValueError("Model not loaded")
            return self.model.get_sentence_embedding_dimension()


##initialize the embedding manager
embedding_manager = EmbeddingManager()
embedding_manager

    

Loading embedding model: all-miniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

model loaded successfully. Embedding dimension: 384


In [70]:
texts = [doc.page_content for doc in chunks]
embeddings = embedding_manager.generateEmbeddings(texts)

Generating embeddings for 31 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (31, 384)


Vector store

In [97]:
class VectorStore:
   """Manages document embeddings in a ChromaDB vector store"""

   def __init__(self, collection_name: str="pdf_documents", persist_directory: str="../data/vector_store"):
      """Initialize the vector store
        Args:
            collection_name: name of the ChromaDB collection
            persist_directory: Directory to persist the vector store"""

      self.collection_name=collection_name
      self.persist_directory = persist_directory
      self.client = None
      self.collection = None
      self._initialize_store()

   def _initialize_store(self):
      """Initialize ChromaDB client and collection"""
      try:
          #Create persistent ChromaDB client 
         os.makedirs(self.persist_directory, exist_ok=True)

         self.client = chromadb.PersistentClient(path=self.persist_directory)

          # Get or create collection
         self.collection = self.client.get_or_create_collection(
            name = self.collection_name,
            metadata={"description":"PDF document embeddings for RAG",
                     "hnsw:space": "cosine"
                     }
         )

         print(f"Vector store initialized. Collection: {self.collection_name}")
         print(f"Existing documents in collection {self.collection.count()}")

      except Exception as e:
         print(f"Error initializing vector store: {e}")
         raise

   def add_documents(self, documents: List[Any], embeddings: np.ndarray):
      """Add documents and their embeddings to the vector store
      
      Args:
         documents: List of langchain documents
         embeddings: Corresponding embeddings for the documents
      """

      if len(documents) != len(embeddings):
         raise ValueError("Number of documents must match number of embeddings")

      print(f"Adding {len(documents)} documents to vector store...")

      # Prepare data for chromaDB

      ids = []
      metadatas = []
      documents_text = []
      embeddings_list = []

      for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
         # Generate unique ID
         doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
         ids.append(doc_id)

         # prepare metadata
         metadata = dict(doc.metadata)
         metadata['doc_index'] = i
         metadata['content_length'] = len(doc.page_content)
         metadatas.append(metadata)

         # Document content
         documents_text.append(doc.page_content)

         # Embedding
         embeddings_list.append(embedding.tolist())

         # Add to Collection
      try:
         self.collection.add(
            ids=ids,
            embeddings=embeddings_list,
            metadatas=metadatas,
            documents=documents_text
         )
         print(f"Successfully added {len(documents)} documents to vector store")
         print(f"Total documents in collection: {self.collection.count()}")

      except Exception as e:
         print(f"Error adding documents to vector store: {e}")
         raise

vectorstore = VectorStore()
vectorstore


Vector store initialized. Collection: pdf_documents
Existing documents in collection 0


In [98]:
vectorstore.add_documents(chunks,embeddings)

Adding 31 documents to vector store...
Successfully added 31 documents to vector store
Total documents in collection: 31


Retriever Pipeline From VectorStore

In [105]:
class RAGRetriever:
    """handles query based retrieval from vector store"""
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """Initialize the retriver
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """

        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.2) -> List[Dict[str,Any]]:
        """
        Retriever relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retriveing documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        # Generate query embeddings
        query_embedding = self.embedding_manager.generateEmbeddings([query])[0]

        ## Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            # Process results
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score(ChromaDB uses cosine distance)
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score':  similarity_score,
                            'distance': distance,
                            'rank': i+1
                        })

                print(f"retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No Documents found")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore, embedding_manager)


 

In [106]:
rag_retriever

In [108]:
results = rag_retriever.retrieve(   "Why does the Transformer use positional encoding?")
print(results)

Retriveing documents for query: 'Why does the Transformer use positional encoding?'
Top K: 5, Score threshold: 0.2
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
retrieved 5 documents (after filtering)
[{'id': 'doc_f36105d6_4', 'content': 'The outputs of the heads are concatenated and projected to produce the multi-head attention output. These\nexamples describe possible behavior, not fixed roles assigned to heads.\n5. Positional information\nSelf-attention by itself does not inherently encode token order. The original Transformer adds positional encodings\nto token embeddings so the model can distinguish different positions.\nThe paper used sine and cosine functions at different frequencies. These encodings provide position-dependent\nsignals, allowing the model to reason about token order and relative offsets.\nToken embeddings answer “what token is this?” Positional information helps answer “where is it in the\nsequence?”\n6. Encoder and decoder architecture\nEncoder\nThe original encoder stacks layers containing multi-head self-attention and a position-wise feed-forward network.\nEach sublayer uses 

Retriveing documents for query: 'How does a Transformer know the order of words?'
Top K: 5, Score threshold: 0.2
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
retrieved 5 documents (after filtering)

Rank: 1
Score: 0.457
Content: The outputs of the heads are concatenated and projected to produce the multi-head attention output. These
examples describe possible behavior, not fixed roles assigned to heads.
5. Positional information
Self-attention by itself does not inherently encode token order. The original Transformer adds p

Rank: 2
Score: 0.445
Content:  The original paper did not introduce every modern LLM design. It introduced the Transformer architecture in an
 encoder–decoder translation setting.
 A Transformer is not automatically an LLM. The architecture, training data, objective, scale, and fine-tuning all
 matter.

Rank: 3
Score: 0.414
Content: AI / ML Study Notes
Page 2
 Scale the scores and apply softmax to get attention weights.
  Use the weights to form a weighted sum of value vectors.
  Pass the updated representations to later layers.
 Example: in “The animal didn’t cross the str

Retriveing documents for query: 'What is the React useEffect hook used for?'
Top K: 5, Score threshold: 0.2
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
retrieved 3 documents (after filtering)
Rank: 1
Score: 0.247
Source: React_Developer_Interview_Guide_Candidate_87.pdf
Content: Question: "The job description emphasizes optimizing components for best performance across web-capable
devices and browsers. Could you walk us through your process for auditing and optimizing a slow React page?
What specific tools (e.g., React DevTo
--------------------------------------------------
Rank: 2
Score: 0.236
Source: React_Developer_Interview_Guide_Candidate_87.pdf
Content: React.js Developer Interview Guide
Candidate Assessment: Candidate-87 (Top Candidate)
Candidate: Candidate-87
Overall Fit Score: 87/100
Role: React.js Developer
Experience: ~10 Years
Education: B.Tech & M.Tech (CS)
Evaluation: Top Candidate
Introduct
--------------------------------------------------
Rank: 3
Score: 0.207
Source: React_Developer_Interview_Guide_Candidate_87.pdf
Content: advanced technical competencies and explore their abi